# Paper MM Demo
Run BTC Up/Down Market Maker in paper mode with real fill simulation.

In [ ]:
import sys
sys.path.insert(0, '/content/ouroboros_repo')

import asyncio
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

from bots.polymarket.run_paper_mm import main

# Run for ~10 minutes with small order size
await main(
    gamma=0.5,
    order_size_usdc=5.0,
    requote_interval=15.0,
)

## Quick Single-Cycle Test
Just compute quotes once without the full loop.

In [ ]:
import asyncio, httpx
from bots.polymarket.scanner import discover_markets, get_orderbook
from bots.polymarket.strategy_btc_mm import BTCUpDownMM, is_updown_market, parse_window_from_question
from bots.polymarket.fair_price import bootstrap_from_klines, get_btc_state
from bots.polymarket.paper_mm import PaperMMEngine
import time

async def test_one_cycle():
    bootstrap_from_klines()
    engine = PaperMMEngine(order_size_usdc=5.0)

    async with httpx.AsyncClient() as client:
        markets = await discover_markets(client)
        for m in markets:
            if not is_updown_market(m.question):
                continue
            window = parse_window_from_question(m.question)
            if not window or window[1] - time.time() < 60:
                continue

            mm = BTCUpDownMM(market=m, clob_client=None, gamma=0.5, order_size_usdc=5.0, paper=True)
            bid, ask = mm.compute_quotes()
            q = mm.last_quotes

            ob_yes = await get_orderbook(client, m.yes_token_id)
            fills = engine.process_quote(m, bid, ask, q.p_fair, ob_yes)

            print(f"\nMarket: {m.question}")
            print(f"  P_fair={q.p_fair:.4f} | bid={bid:.4f} ask={ask:.4f} spread={q.spread:.4f}")
            print(f"  Market: best_bid={ob_yes.best_bid():.4f} best_ask={ob_yes.best_ask():.4f}")
            print(f"  Fills this cycle: {len(fills)}")
            break

    engine.print_summary()

await test_one_cycle()